This project usus Python 3.10.20, Tensorflow 2.12.0, numpy 1.23.5, and mediapipe 0.10.9

```pip install tensorflow==2.12.0 numpy==1.23.5 mediapipe==0.10.9``` <-- to use in wsl2

In [ ]:
import os
import shutil
import kagglehub
from pathlib import Path

target_dir = "../fall-video-dataset"

# Check if dataset already exists
if Path(target_dir).exists() and any(Path(target_dir).iterdir()):
    print("Dataset already exists, skipping download.")
else:
    print("Downloading dataset...")
    path = kagglehub.dataset_download("payutch/fall-video-dataset")
    print("Path to dataset files:", path)

    os.makedirs(target_dir, exist_ok=True)

    for item in os.listdir(path):
        s = os.path.join(path, item)
        d = os.path.join(target_dir, item)

        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)

    print("Dataset copied to:", target_dir)

In [ ]:
import tensorflow as tf

Below is the short description as provided in the kaggle page we downloaded the dataset from Kaggle

```About this directory```

```-Raw_Video directory contains raw video footage of human fall.```

```-Keypoints_CSV directory```

```-For each input video (e.g., video1.mp4 in Fall/Raw_Video/), the script generates a corresponding CSV file (e.g., video1_keypoints.csv in Fall/Keypoints_CSV/).```

```-Each CSV file contains frame-by-frame data of the 17 extracted keypoints, their 2D coordinates, and confidence scores.```

```-The CSV file will have columns: "Frame", "Keypoint", "X", "Y", "Confidence".```





Referencing the dataset description on the kaggle page, we load the CSV into frame vectors

Load and Preprocess the CSV's


In [ ]:
import pandas as pd
import numpy as np 
from pathlib import Path

def load_video_keypoints(csv_path):
    df = pd.read_csv(csv_path)
    frames = sorted(df['Frame'].unique())
    keypoints_per_frame = []

    for frame in frames:
        frame_data = df[df['Frame'] == frame].sort_values('Keypoint')
        # Shape: (17, 3) — x, y, confidence
        kp = frame_data[['X', 'Y', 'Confidence']].values
        keypoints_per_frame.append(kp)

    return np.array(keypoints_per_frame)  # (T, 17, 3)

Build a Sliding Window Dataset, using 30 frame windows

In [ ]:
def create_sequences(keypoints, label, window_size=30, stride=15):
    sequences, labels = [], []
    T = len(keypoints)
    for start in range(0, T - window_size + 1, stride):
        seq = keypoints[start:start + window_size]  # (30, 17, 3)
        sequences.append(seq)
        labels.append(label)
    return sequences, labels

# Build full dataset
def build_dataset(fall_csv_dir, non_fall_csv_dir, window_size=30):
    X, y = [], []
    for path in Path(fall_csv_dir).glob("*.csv"):
        kp = load_video_keypoints(path)
        seqs, labs = create_sequences(kp, label=1, window_size=window_size)
        X.extend(seqs); y.extend(labs)

    for path in Path(non_fall_csv_dir).glob("*.csv"):
        kp = load_video_keypoints(path)
        seqs, labs = create_sequences(kp, label=0, window_size=window_size)
        X.extend(seqs); y.extend(labs)

    return np.array(X), np.array(y)  # (N, 30, 17, 3)

Normalize the Data and save as a .npz file

In [ ]:
def normalize_keypoints(X):
    # X shape: (N, T, 17, 3)
    xy = X[..., :2]  # (N, T, 17, 2)
    conf = X[..., 2:3]

    min_xy = xy.min(axis=2, keepdims=True)  # per frame min
    max_xy = xy.max(axis=2, keepdims=True)
    range_xy = np.where((max_xy - min_xy) == 0, 1, max_xy - min_xy)

    xy_norm = (xy - min_xy) / range_xy  # → [0, 1]
    return np.concatenate([xy_norm, conf], axis=-1)


# Run and saved once as sa compressed npz file to quicken training
# X, y = build_dataset(r"..\fall-video-dataset\Fall\Keypoints_CSV", r"..\fall-video-dataset\No_Fall\Keypoints_CSV")
# X = normalize_keypoints(X)

# np.savez_compressed("dataset_cache.npz", X=X, y=y)
data = np.load("dataset_cache.npz")
X, y = data['X'], data['y']

Add derived features (velocity and acceleration)

In [ ]:
def add_derived_features(X):
    # X shape: (N, T, 17, 3)
    xy = X[..., :2]
    conf = X[..., 2:3]

    # Velocity — joint movement between frames
    velocity = np.diff(xy, axis=1, prepend=xy[:, :1])      # (N, T, 17, 2)

    # Acceleration — change in velocity
    acceleration = np.diff(velocity, axis=1, prepend=velocity[:, :1])  # (N, T, 17, 2)

    # Spine angle — angle between hip midpoint and shoulder midpoint
    hip_mid      = (xy[:, :, 11] + xy[:, :, 12]) / 2       # (N, T, 2)
    shoulder_mid = (xy[:, :, 5]  + xy[:, :, 6])  / 2
    spine_vec    = shoulder_mid - hip_mid
    spine_angle  = np.arctan2(spine_vec[..., 0], spine_vec[..., 1])[..., np.newaxis]  # (N, T, 1)
    spine_angle  = np.repeat(spine_angle[:, :, np.newaxis, :], 17, axis=2)            # (N, T, 17, 1)

    # Stack all features: x, y, conf, vx, vy, ax, ay, spine_angle = 8 features
    return np.concatenate([xy, conf, velocity, acceleration, spine_angle], axis=-1)   # (N, T, 17, 8)

X = add_derived_features(X)  # run before train/val/test split
# Update model: n_features=8

Train/Validate/Split, so we dont have to make our own training dataset

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, stratify=y_train, random_state=42)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Data Augmentation

In [ ]:
def augment_batch(X_batch, y_batch):
    augmented_X, augmented_y = [], []

    for seq, label in zip(X_batch, y_batch):
        augmented_X.append(seq)
        augmented_y.append(label)

        # Horizontal flip
        flipped = seq.copy()
        flipped[..., 0] = 1 - flipped[..., 0]
        augmented_X.append(flipped)
        augmented_y.append(label)

        # Gaussian noise
        noisy = seq + np.random.normal(0, 0.01, seq.shape)
        augmented_X.append(noisy)
        augmented_y.append(label)

        # Time reversal — reversed motion still looks like a fall
        augmented_X.append(seq[::-1])
        augmented_y.append(label)

    return np.array(augmented_X), np.array(augmented_y)

X_train, y_train = augment_batch(X_train, y_train)

Build CNN (Convolutional Neural Networks) + LSTM (Long Short-Term Memory networks) model 

In [ ]:
from keras import layers

def build_cnn_lstm_attention(window_size=30, n_keypoints=17, n_features=8):
    inputs = tf.keras.Input(shape=(window_size, n_keypoints, n_features))

    # CNN per frame
    x = layers.TimeDistributed(layers.Conv1D(64, 3, activation='relu', padding='same'))(inputs)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    x = layers.TimeDistributed(layers.Conv1D(128, 3, activation='relu', padding='same'))(x)
    x = layers.TimeDistributed(layers.Conv1D(256, 3, activation='relu', padding='same'))(x)
    x = layers.TimeDistributed(layers.GlobalAveragePooling1D())(x)      # (N, T, 256)

    # Bidirectional LSTM — looks at motion forwards AND backwards
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.3))(x)

    # Attention — focus on the most important frames
    attention  = layers.Dense(1, activation='tanh')(x)                  # (N, T, 1)
    attention  = layers.Flatten()(attention)                             # (N, T)
    attention  = layers.Activation('softmax')(attention)
    attention  = layers.RepeatVector(256)(attention)                     # (N, 256, T)
    attention  = layers.Permute([2, 1])(attention)                       # (N, T, 256)
    x          = layers.Multiply()([x, attention])
    x          = layers.Lambda(lambda t: tf.reduce_sum(t, axis=1))(x)   # (N, 256)

    # Classifier
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    return tf.keras.Model(inputs, outputs)

model = build_cnn_lstm_attention()
model.summary()

Fix class imbalance

In [ ]:
from sklearn.utils.class_weight import compute_class_weight



class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

Compile and Train the model

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_auc', mode='max'),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5),
    tf.keras.callbacks.ModelCheckpoint("best_fall_detection_model.keras", save_best_only=True, monitor='val_auc', mode='max')
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    class_weight=class_weight_dict,   # class imbalance fix
    callbacks=callbacks
)

Evaulate the model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()

print(classification_report(y_test, y_pred, target_names=['Non-Fall', 'Fall']))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['Non-Fall','Fall'], yticklabels=['Non-Fall','Fall'])
plt.title("Confusion Matrix"); plt.show()

Install required libraries for next step

```pip install opencv-python mediapipe tensorflow numpy```

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import mediapipe as mp
from collections import deque
import warnings
warnings.filterwarnings("ignore", category=Warning)

# --- Config ---
MODEL_PATH        = r"./best_fall_detection_model.keras"
WINDOW_SIZE       = 30      # must match what you trained with
THRESHOLD         = 0.90    # raised to reduce false positives
COOLDOWN_SEC      = 3       # seconds before another alert can trigger
CONFIRM_THRESHOLD = 5       # consecutive predictions needed before alerting
MIN_VISIBILITY    = 0.5     # minimum keypoint confidence
MIN_LEGS_VISIBLE  = 2       # minimum visible leg keypoints required

# Leg keypoint indices (hips, knees, ankles)
LEG_KEYPOINTS = [11, 12, 13, 14, 15, 16]

# --- Load model ---
model = tf.keras.models.load_model(MODEL_PATH)

# --- Mediapipe pose ---
from mediapipe.python.solutions import pose as mp_pose
from mediapipe.python.solutions import drawing_utils as mp_drawing
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

# --- State ---
frame_buffer  = deque(maxlen=WINDOW_SIZE)
prob_history  = deque(maxlen=10)   # smooth predictions over last 10 frames
cooldown      = 0
fall_detected = False
fall_counter  = 0
prob          = 0.0                # initialise so overlay never crashes

def extract_keypoints(results):
    """Extract 17 keypoints — returns None if leg keypoints aren't visible."""
    if not results.pose_landmarks:
        return None

    landmarks = results.pose_landmarks.landmark[:17]

    # Check leg visibility
    leg_visibilities  = [landmarks[i].visibility for i in LEG_KEYPOINTS if i < len(landmarks)]
    visible_leg_count = sum(v >= MIN_VISIBILITY for v in leg_visibilities)

    if visible_leg_count < MIN_LEGS_VISIBLE:
        return None   # not enough legs visible — skip frame

    keypoints = []
    for lm in landmarks:
        keypoints.append([lm.x, lm.y, lm.visibility])

    return np.array(keypoints)  # (17, 3)

def normalize_keypoints(seq):
    """Same normalization used during training."""
    seq      = np.array(seq)        # (T, 17, 3)
    xy       = seq[..., :2]
    conf     = seq[..., 2:3]
    min_xy   = xy.min(axis=1, keepdims=True)
    max_xy   = xy.max(axis=1, keepdims=True)
    range_xy = np.where((max_xy - min_xy) == 0, 1, max_xy - min_xy)
    xy_norm  = (xy - min_xy) / range_xy
    return np.concatenate([xy_norm, conf], axis=-1)  # (T, 17, 3)

def predict_fall(frame_buffer):
    """Run model inference on current buffer."""
    seq  = normalize_keypoints(list(frame_buffer))  # (T, 17, 3)
    seq  = np.expand_dims(seq, axis=0)              # (1, T, 17, 3)
    prob = model.predict(seq, verbose=0)[0][0]
    return float(prob)

# --- Webcam loop ---
cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS) or 30

print("Starting webcam... Press Q to quit.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    h, w = frame.shape[:2]

    # --- Pose estimation ---
    rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb)

    # --- Extract keypoints ---
    keypoints     = extract_keypoints(results)
    legs_visible  = keypoints is not None

    if keypoints is not None:
        frame_buffer.append(keypoints)
    else:
        frame_buffer.clear()    # wipe buffer if legs not visible
        prob_history.clear()
        fall_counter = 0

    # --- Run model when buffer is full ---
    if len(frame_buffer) == WINDOW_SIZE and cooldown == 0:
        raw_prob = predict_fall(frame_buffer)
        prob_history.append(raw_prob)
        prob = float(np.mean(prob_history))   # smoothed probability

        if prob >= THRESHOLD:
            fall_counter += 1
            if fall_counter >= CONFIRM_THRESHOLD:
                fall_detected = True
                cooldown      = int(COOLDOWN_SEC * fps)
                fall_counter  = 0
                prob_history.clear()
                print(f"!! FALL DETECTED (confidence: {prob:.2f})")
        else:
            fall_counter = 0   # reset if prediction drops

    # --- Draw skeleton ---
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(
            frame,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
            mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2)
        )

    # --- Overlay UI ---
    bar_color = (0, 0, 255) if fall_detected else (0, 255, 0)

    if fall_detected:
        label = "!! FALL DETECTED"
    elif not legs_visible:
        label = "Legs not visible - paused"
        bar_color = (0, 165, 255)   # orange when paused
    else:
        label = "Monitoring..."

    prob_text = f"Fall prob: {prob:.2f}" if len(frame_buffer) == WINDOW_SIZE else "Buffering..."
    confirm_text = f"Confirm: {fall_counter}/{CONFIRM_THRESHOLD}"

    cv2.rectangle(frame, (0, 0), (w, 70), (0, 0, 0), -1)
    cv2.putText(frame, label,        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, bar_color, 2)
    cv2.putText(frame, prob_text,    (10, 52), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(frame, confirm_text, (10, 67), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200, 200, 200), 1)

    # Buffer fill indicator at bottom
    buf_pct = len(frame_buffer) / WINDOW_SIZE
    cv2.rectangle(frame, (0, h - 10), (int(w * buf_pct), h), (0, 165, 255), -1)

    cv2.imshow("Fall Detection", frame)

    # --- Cooldown tick ---
    if cooldown > 0:
        cooldown -= 1
        if cooldown == 0:
            fall_detected = False
            print("Monitoring resumed...")

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
pose.close()